In [ ]:
import fastf1
import os

os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

print("Setup fertig!")

In [ ]:
session = fastf1.get_session(2024, "Monza", "Q")
session.load()

laps = session.laps
print(f"Runden geladen: {len(laps)}")

In [ ]:
import matplotlib.pyplot as plt

# Schnellste Runde pro Fahrer
fastest_laps = laps.pick_quicklaps().groupby("Driver")["LapTime"].min().sort_values()

# Umrechnen in Sekunden für lesbare Achse
fastest_seconds = fastest_laps.dt.total_seconds()

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(fastest_seconds.index, fastest_seconds.values - fastest_seconds.min())
ax.set_xlabel("Zeit-Rückstand zur Pole (Sekunden)")
ax.set_title("Monza 2024 Qualifying – Rückstand zur Pole Position")
plt.tight_layout()
plt.show()

In [ ]:
# Teamfarben laden
fastf1.plotting.setup_mpl(mpl_timedelta_support=True, color_scheme=None)

fastest_laps = laps.pick_quicklaps().groupby("Driver")["LapTime"].min().sort_values()
fastest_seconds = fastest_laps.dt.total_seconds()

# Farbe pro Fahrer
colors = []
for driver in fastest_seconds.index:
    team = laps[laps["Driver"] == driver]["Team"].iloc[0]
    color = fastf1.plotting.get_team_color(team, session=session)
    colors.append(color)

# Plot mit Teamfarben
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(fastest_seconds.index, fastest_seconds.values - fastest_seconds.min(), color=colors)
ax.set_xlabel("Zeit-Rückstand zur Pole (Sekunden)")
ax.set_title("Monza 2024 Qualifying – Rückstand zur Pole Position")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
import fastf1.plotting

# Teamfarben laden
fastf1.plotting.setup_mpl(mpl_timedelta_support=True, color_scheme=None)

fastest_laps = laps.pick_quicklaps().groupby("Driver")["LapTime"].min().sort_values()
fastest_seconds = fastest_laps.dt.total_seconds()

# Farbe pro Fahrer
colors = []
for driver in fastest_seconds.index:
    team = laps[laps["Driver"] == driver]["Team"].iloc[0]
    color = fastf1.plotting.get_team_color(team, session=session)
    colors.append(color)

# Plot mit Teamfarben
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(fastest_seconds.index, fastest_seconds.values - fastest_seconds.min(), color=colors)
ax.set_xlabel("Zeit-Rückstand zur Pole (Sekunden)")
ax.set_title("Monza 2024 Qualifying – Rückstand zur Pole Position")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Schnellste Runde von Norris und Leclerc
nor = laps.pick_drivers("NOR").pick_fastest()
lec = laps.pick_drivers("LEC").pick_fastest()

# Telemetrie laden
tel_nor = nor.get_telemetry()
tel_lec = lec.get_telemetry()

# Plot: Geschwindigkeit über Distanz
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(tel_nor["Distance"], tel_nor["Speed"], 
        color=fastf1.plotting.get_team_color("McLaren", session=session), 
        label="Norris (McLaren)")

ax.plot(tel_lec["Distance"], tel_lec["Speed"], 
        color=fastf1.plotting.get_team_color("Ferrari", session=session), 
        label="Leclerc (Ferrari)")

ax.set_xlabel("Distanz (m)")
ax.set_ylabel("Geschwindigkeit (km/h)")
ax.set_title("Monza 2024 Qualifying – Speed Trace: Norris vs. Leclerc")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import fastf1.plotting

# Load telemetry for the fastest laps of Norris and Leclerc
nor = laps.pick_drivers("NOR").pick_fastest()
lec = laps.pick_drivers("LEC").pick_fastest()

tel_nor = nor.get_telemetry()
tel_lec = lec.get_telemetry()

# Plot speed, throttle and brake traces side by side
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

color_nor = fastf1.plotting.get_team_color("McLaren", session=session)
color_lec = fastf1.plotting.get_team_color("Ferrari", session=session)

# Speed
axes[0].plot(tel_nor["Distance"], tel_nor["Speed"], color=color_nor, label="Norris")
axes[0].plot(tel_lec["Distance"], tel_lec["Speed"], color=color_lec, label="Leclerc")
axes[0].set_ylabel("Speed (km/h)")
axes[0].legend()

# Throttle
axes[1].plot(tel_nor["Distance"], tel_nor["Throttle"], color=color_nor, label="Norris")
axes[1].plot(tel_lec["Distance"], tel_lec["Throttle"], color=color_lec, label="Leclerc")
axes[1].set_ylabel("Throttle (%)")

# Brake
axes[2].plot(tel_nor["Distance"], tel_nor["Brake"], color=color_nor, label="Norris")
axes[2].plot(tel_lec["Distance"], tel_lec["Brake"], color=color_lec, label="Leclerc")
axes[2].set_ylabel("Brake")
axes[2].set_xlabel("Distance (m)")

fig.suptitle("Monza 2024 Qualifying – Telemetry Comparison: Norris vs. Leclerc", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compare ERS deployment across top teams on a power circuit
teams = {
    "NOR": "McLaren",
    "LEC": "Ferrari", 
    "HAM": "Mercedes"
}

fig, ax = plt.subplots(figsize=(14, 6))

for driver, team in teams.items():
    fastest = laps.pick_drivers(driver).pick_fastest()
    tel = fastest.get_telemetry()
    color = fastf1.plotting.get_team_color(team, session=session)
    ax.plot(tel["Distance"], tel["RPM"], color=color, label=f"{driver} ({team})")

ax.set_xlabel("Distance (m)")
ax.set_ylabel("RPM")
ax.set_title("Monza 2024 Qualifying – Engine RPM Comparison (ERS proxy)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Speed vs RPM to visualise ERS boost zones per team
teams = {
    "NOR": "McLaren",
    "LEC": "Ferrari",
    "HAM": "Mercedes"
}

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for i, (driver, team) in enumerate(teams.items()):
    fastest = laps.pick_drivers(driver).pick_fastest()
    tel = fastest.get_telemetry()
    color = fastf1.plotting.get_team_color(team, session=session)

    # Speed
    axes[i].plot(tel["Distance"], tel["Speed"], color=color, label="Speed (km/h)")
    axes[i].set_ylabel("Speed (km/h)")
    axes[i].set_title(f"{driver} – {team}")
    axes[i].legend(loc="upper left")

    # RPM on secondary axis
    ax2 = axes[i].twinx()
    ax2.plot(tel["Distance"], tel["RPM"], color=color, alpha=0.4, linestyle="--", label="RPM")
    ax2.set_ylabel("RPM")
    ax2.legend(loc="upper right")

axes[-1].set_xlabel("Distance (m)")
fig.suptitle("Monza 2024 Qualifying – Speed & RPM: ERS Deployment Zones", fontsize=14)
plt.tight_layout()
plt.show()